# Ship Detection — YOLO-OBB baseline

Fine-tunes a pretrained YOLO oriented-bounding-box (OBB) model on `data/roboflow_dataset/` to detect ships (including berthed ones) in port satellite imagery. OBB instead of a regular axis-aligned box because berthed ships sit at an angle relative to the image.

Run the cells top to bottom.

## 1. Install dependencies

In [1]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load pretrained model

`yolo11n-obb.pt` is pretrained on DOTA (an aerial/oriented-object dataset), so it already has a head start on "objects seen from above, at an angle" before we fine-tune it on our ships.

In [ ]:
from ultralytics import YOLO

DATA_YAML = "data/roboflow_dataset/data.yaml"
model = YOLO("yolo11n-seg.pt")

Creating new Ultralytics Settings v0.0.7 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\owner\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 3. Train

Only 26 annotated images right now, so this is a baseline sanity check, not a final model. `imgsz=1024` matches the tile size the images were cut at.

In [3]:
results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=1024,
    project="runs",
    name="ship-obb-baseline",
)

Ultralytics 8.4.118  Python-3.14.6 torch-2.13.0+cpu CPU (11th Gen Intel Core i5-11400H @ 2.70GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/roboflow_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ship-obb-b

TypeError: ERROR ❌ OBB dataset incorrectly formatted or not a OBB dataset.
This error can occur when incorrectly training a 'OBB' model on a 'detect' dataset, i.e. 'yolo train model=yolo26n-obb.pt data=dota8.yaml'.
Verify your dataset is a correctly formatted 'OBB' dataset using 'data=dota8.yaml' as an example.
See https://docs.ultralytics.com/datasets/obb/ for help.

## 4. Evaluate on the test split

In [ ]:
metrics = model.val(data=DATA_YAML, split="test")
metrics

## 5. Visualize a prediction

In [ ]:
import glob

test_images = glob.glob("data/roboflow_dataset/test/images/*.jpg")
pred = model.predict(test_images[0], save=False)[0]
pred.show()